In [1]:
import numpy as np
from scipy.integrate import fixed_quad, quad
import os
import plotly.graph_objects as go
from scipy.special import j0, jv
import pandas as pd

In [2]:
# Leitura do arquivo com separação por espaços
data_atlas = pd.read_csv(
    "../../../data/sigma_tot_2/ensemble_StRh_atlas.dat",
    delim_whitespace=True,
    header=None,
    nrows=70  # lê apenas as 70 primeiras linhas
)

x_atlas = data_atlas[0].to_numpy()
y_atlas = data_atlas[1].to_numpy()
y_error_atlas = data_atlas[2].to_numpy()

In [3]:
# === Global Configuration and Constants ===
start_sqrt_s = 1  # Global parameter controlling energy scale
b_0 = (33 - 6) / (12 * np.pi)  # β0 for nf=3
Lambda = 0.284  # ΛQCD in GeV
gamma_1 = 0.084
gamma_2 = 2.36
rho = 4.0

sigma_tot_lst = []
sqrt_s_lst = []
error_lst = []

s0 = 1.0  # GeV^2

epsilon_atlas = 0.0729

model_params = {
    'atlas': {
        'pl':  {'mg': 0.412, 'a1': 1.652, 'a2': 1.479}
    }
}

epsilon_values = {
    'atlas': epsilon_atlas
}


In [4]:

# === Auxiliary Functions for Physical Model ===
def m2_pl(q2, mg):
    lambda_squared = Lambda ** 2
    rho_mg_squared = rho * mg ** 2
    ratio = np.log((q2 + rho_mg_squared) / lambda_squared) / np.log(rho_mg_squared / lambda_squared)
    return (mg ** 4 / (q2 + mg ** 2)) * ratio ** (gamma_2 - 1)

def get_m2_function(mass_model):
    return m2_pl

def G_p(q2, a1, a2):
    return np.exp(-(a1 * q2 + a2 * q2 ** 2))

def alpha_D(q2, mg, m2_func):
    m2 = m2_func(q2, mg)
    return 1.0 / (b_0 * (q2 + m2) * np.log((q2 + 4 * m2) / (Lambda ** 2)))

def T_1(k, q, phi, mg, a1, a2, m2_func):
    q2 = q ** 2
    qk_cos = q * k * np.cos(phi)
    qk_plus_squared = q2 / 4 + qk_cos + k ** 2
    qk_minus_squared = q2 / 4 - qk_cos + k ** 2

    alpha_D_plus = alpha_D(qk_plus_squared, mg, m2_func)
    alpha_D_minus = alpha_D(qk_minus_squared, mg, m2_func)
    G0 = G_p(q2, a1, a2)

    return alpha_D_plus * alpha_D_minus * G0 ** 2

def T_2(k, q, phi, mg, a1, a2, m2_func):
    q2 = q ** 2
    qk_cos = q * k * np.cos(phi)
    qk_plus_squared = q2 / 4 + qk_cos + k ** 2
    qk_minus_squared = q2 / 4 - qk_cos + k ** 2

    alpha_D_plus = alpha_D(qk_plus_squared, mg, m2_func)
    alpha_D_minus = alpha_D(qk_minus_squared, mg, m2_func)

    factor = q2 + 9 * abs(k ** 2 - q2 / 4)

    G0 = G_p(q2, a1, a2)
    G_minus = G_p(factor, a1, a2)

    return alpha_D_plus * alpha_D_minus * G_minus * (2 * G0 - G_minus)

def integrand(y, x, mg, a1, a2, m2_func):
    k = sqrt_s * x
    phi = 2 * np.pi * y
    jacobian = 2 * np.pi * sqrt_s

    return k * (T_1(k, 0.0, phi, mg, a1, a2, m2_func) - T_2(k, 0.0, phi, mg, a1, a2, m2_func)) * jacobian

def amp_calculation(diff_T, s, epsilon):
    alpha_pomeron = 1.0 + epsilon
    regge_factor = (s / s0) ** alpha_pomeron
    
    return 1j * 8.0 * regge_factor * diff_T

def sigma_tot(amp_value, s):
    return amp_value.imag / s * 0.389379323


In [5]:
amp_born_lst = []

sqrt_s_lst = []

# === Main Function ===
def main():
    global start_sqrt_s
    global sqrt_s

    max_sqrt_s = 13000
    step = 100
    n_points = 10000

    # Using only PL model with ATLAS
    mass_model = 'pl'
    ensemble = 'atlas'

    fig = go.Figure()

    sigma_tot_lst = []
    

    

    m2_func = get_m2_function(mass_model)
    params = model_params[ensemble][mass_model]
    mg, a1, a2 = params['mg'], params['a1'], params['a2']
    epsilon = epsilon_values[ensemble]

    sqrt_s = start_sqrt_s
    while sqrt_s <= max_sqrt_s:
        def inner_integral(x):
            return fixed_quad(
                lambda y: integrand(y, x, mg, a1, a2, m2_func),
                0, 1,
                n=n_points
            )[0]

        integral_value = fixed_quad(
            inner_integral,
            0, 1,
            n=n_points
        )[0]

        diff_T = integral_value
        s = sqrt_s * sqrt_s

        amp_value = amp_calculation(diff_T, s, epsilon)
        sigma_tot_value = sigma_tot(amp_value, s)

        sigma_tot_lst.append(sigma_tot_value)
        sqrt_s_lst.append(sqrt_s)
        amp_born_lst.append(amp_value)

        sqrt_s += step

    # Add PL model trace
    fig.add_trace(go.Scatter(
        x=sqrt_s_lst,
        y=sigma_tot_lst,
        mode='lines+markers',
        line=dict(
            color='blue',
            width=2
        ),
        marker=dict(
            size=4
        ),
        name='PL Model (ATLAS)'
    ))

    # Add ATLAS data
    fig.add_trace(go.Scatter(
        x=x_atlas,
        y=y_atlas,
        mode='markers',
        marker=dict(
            color='black',
            size=6,
            symbol='square'
        ),
        error_y=dict(
            type='data',
            array=y_error_atlas,
            visible=True
        ),
        name='ATLAS Data'
    ))

    # Configure layout
    fig.update_layout(
        title='Sigma Tot vs. sqrt(s) - PL Model with ATLAS Data',
        xaxis=dict(
            title='sqrt(s) [GeV]',
            type='log',
        ),
        yaxis=dict(
            title='Sigma Tot [mb]',
        ),
        showlegend=True,
        legend=dict(
            title='Model/Data'
        ),
        plot_bgcolor='white',
        hovermode='x unified'
    )
    
    fig.update_xaxes(gridcolor='lightgray')
    fig.update_yaxes(gridcolor='lightgray')

    # fig.show(renderer="browser")
    # fig.write_html("results/sigma_tot/sigma_tot_pl_atlas.html")
    # fig.write_image("results/sigma_tot/sigma_tot_pl_atlas.pdf", width=1200, height=600)


In [6]:
if __name__ == "__main__":
    main()

In [7]:
lst_s = []
for key, value in enumerate(sqrt_s_lst):
    lst_s.append(value ** 2)
    # print(f"sqrt(s) = {value:.2f} GeV, s = {s_lst[key]:.2f} GeV^2")

lst_b = np.linspace(0, 30, 130)


def int_chi(b, q, amp):
    return q * j0(b * q) * amp 


def int_amp_eik(b, chi):
    return b * (1 - np.exp(1j * chi))


def sigma_eik(s, amp):
    return (4*np.pi)/s * amp.imag * 0.389379323


In [8]:
upper_limit = 10 
n = 10

def chi(b, amp_born, s):

    pre_factor = 1/s

    int_chi_val = fixed_quad(
        lambda q: int_chi(b, q, amp_born),
        0, upper_limit, n=n
    )[0]

    chi_val = pre_factor * int_chi_val

    return chi_val

def amp_eik(s, amp, chi):

    pre_factor = 1j * s

    int_amp_eik_val = fixed_quad(
        lambda b: int_amp_eik(b, chi(b, amp, s)), 
    0, upper_limit, n = n)[0]

    amp_eik_val = pre_factor * int_amp_eik_val 

    return amp_eik_val

lst_amp_eik = []

for amp_born_val, s_val in zip(amp_born_lst, lst_s):
    amp_eik_val = amp_eik(s_val, amp_born_val, chi)
    # print(amp_eik_val)
    lst_amp_eik.append(amp_eik_val)

# print(f"{lst_amp_eik[-1].imag:.2e}")  

for amp_eik_val, s_val in zip(lst_amp_eik, lst_s):
    print(sigma_eik(s_val, amp_eik_val))

244.65424411931343
244.65424411931343
244.65424411931343
244.65424411931343
244.65424411931343
244.65424411931343
244.65424411931343
244.65424411931343
244.6542441193134
244.65424411931343
244.65424411931343
244.6542441193134
244.65424411931343
244.65424411931343
244.65424411931343
244.6542441193134
244.65424411931343
244.65424411931343
244.65424411931343
244.6542441193134
244.65424411931343
244.6542441193134
244.65424411931343
244.65424411931343
244.65424411931343
244.6542441193134
244.6542441193134
244.65424411931343
244.65424411931343
244.65424411931343
244.65424411931343
244.65424411931343
244.6542441193134
244.65424411931343
244.65424411931343
244.65424411931343
244.65424411931343
244.65424411931343
244.65424411931343
244.65424411931343
244.65424411931343
244.65424411931343
244.65424411931343
244.65424411931343
244.6542441193134
244.65424411931343
244.65424411931343
244.65424411931343
244.65424411931343
244.65424411931343
244.65424411931343
244.65424411931343
244.65424411931343
24

In [9]:

# print(sigma_eik(13000**2, 1j*3.5*(10**9))) -> 101.33607744586946

In [10]:
import numpy as np
from scipy.special import j0
from scipy.integrate import fixed_quad

# Global integration limit
upper_limit = 10  

def compute_integral(s, amp):

    
    def chi(s, b):

        def chi_integrand(q):
            return q * j0(b * q)
        
        chi_int_value, _ = fixed_quad(chi_integrand, 0, upper_limit, n = 10)

        chi_value = (1/s) * amp * chi_int_value

        return chi_value

    def amp_eik_integrand(b):
        chi_val = chi(s, b)
        return b * (1 - np.exp(1j * chi_val))

    # Integrate the complex-valued function directly
    result, _ = fixed_quad(amp_eik_integrand, 0, upper_limit, n=10)
    return 1j * s * result

lst_amp_eik = []

for amp_born_value, s_value in zip(amp_born_lst, lst_s):
    amp_eik_val = compute_integral(s_value, amp_born_value)

    print(amp_eik_val)
    lst_amp_eik.append(amp_eik_val)

lst_sigma_eik = []

for lst_amp_eik_value, s_value in zip(lst_amp_eik, lst_s):
    sigma_eik_value = sigma_eik(s_value, lst_amp_eik_value)
    # print(sigma_eik_value)



50j
510050j
2020050j
4530050j
8040050j
12550050j
18060050j
24570050j
32080050j
40590050j
50100050j
60610050j
72120050j
84630050j
98140050j
112650050j
128160050j
144670050j
162180050j
180690050j
200200050j
220710050j
242220050j
264730050j
288240050j
312750050j
338260050j
364770050j
392280050j
420790050j
450300050j
480810050j
512320050j
544830050j
578340050j
612850050j
648360050j
684870050j
722380050j
760890050j
800400050j
840910050j
882420050j
924930050j
968440050j
1012950050j
1058460050j
1104970050j
1152480050j
1200990050j
1250500050j
1301010050j
1352520050j
1405030050j
1458540050j
1513050050j
1568560050j
1625070050j
1682580050j
1741090050j
1800600050j
1861110050j
1922620050j
1985130050j
2048640050j
2113150050j
2178660050j
2245170050j
2312680050j
2381190050j
2450700050j
2521210050j
2592720050j
2665230050j
2738740050j
2813250050j
2888760050j
2965270050j
3042780050j
3121290050j
3200800050j
3281310050j
3362820050j
3445330050j
3528840050j
3613350050j
3698860050j
3785370050j
3872880050j
396